In [1]:
import json
import math
import os
import functions_for_experiments
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3538.01it/s]


In [ ]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [3]:
n = len(golden_data)

## Baseline

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COLLECTION_OPENAI = "ucu_documents_openai"
openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [5]:
total_hit_openai, total_mrr_openai, total_recall_openai, total_ndcg_openai, \
total_hit_reranked_openai, total_mrr_reranked_openai, total_recall_reranked_openai, total_ndcg_reranked_openai = functions_for_experiments.get_metrics(None, COLLECTION_OPENAI, openai_client=openai_client)

In [6]:
print("Results for OpenAI model:")
print(f"Hit Rate @ 5: {round(total_hit_openai/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_openai/n, 2)}")
print(f"Recall @ 5: {round(total_recall_openai/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_openai/n, 2)}")

Results for OpenAI model:
Hit Rate @ 5: 0.84
MRR @ 5: 0.61
Recall @ 5: 0.75
NDCG @ 5: 0.61


In [7]:
print("Results for OpenAI model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai/n, 2)}")

Results for OpenAI model:
Hit Rate @ 5: 0.86
MRR @ 5: 0.79
Recall @ 5: 0.79
NDCG @ 5: 0.76


## HyDE

In [5]:
total_hit_openai_hyde, total_mrr_openai_hyde, total_recall_openai_hyde, total_ndcg_openai_hyde, \
total_hit_reranked_openai_hyde, total_mrr_reranked_openai_hyde, total_recall_reranked_openai_hyde, total_ndcg_reranked_openai_hyde = functions_for_experiments.get_metrics_hyde(None, COLLECTION_OPENAI, openai_client=openai_client)

In [6]:
print("Results for OpenAI model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_openai_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_openai_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_openai_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_openai_hyde/n, 2)}")

Results for OpenAI model (HyDE):
Hit Rate @ 5: 0.68
MRR @ 5: 0.48
Recall @ 5: 0.6
NDCG @ 5: 0.48


In [7]:
print("Results for OpenAI model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_hyde/n, 2)}")

Results for OpenAI model (HyDE):
Hit Rate @ 5: 0.82
MRR @ 5: 0.75
Recall @ 5: 0.74
NDCG @ 5: 0.71


## Query Transform

In [8]:
total_hit_openai_transform, total_mrr_openai_transform, total_recall_openai_transform, total_ndcg_openai_transform, \
total_hit_reranked_openai_transform, total_mrr_reranked_openai_transform, total_recall_reranked_openai_transform, total_ndcg_reranked_openai_transform = functions_for_experiments.get_metrics_query_transform(None, COLLECTION_OPENAI, openai_client=openai_client)

In [9]:
print("Results for OpenAI model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_openai_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_openai_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_openai_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_openai_transform/n, 2)}")

Results for OpenAI model (Query Transform):
Hit Rate @ 5: 0.76
MRR @ 5: 0.56
Recall @ 5: 0.66
NDCG @ 5: 0.55


In [10]:
print("Results for OpenAI model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_transform/n, 2)}")

Results for OpenAI model (Query Transform):
Hit Rate @ 5: 0.84
MRR @ 5: 0.77
Recall @ 5: 0.77
NDCG @ 5: 0.73


## Hybrid

In [11]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [15]:
total_hit_openai_sparse, total_mrr_openai_sparse, total_recall_openai_sparse, total_ndcg_openai_sparse, \
total_hit_reranked_openai_sparse, total_mrr_reranked_openai_sparse, total_recall_reranked_openai_sparse, total_ndcg_reranked_openai_sparse = functions_for_experiments.get_metrics_sparse(None, COLLECTION_OPENAI, sparse_model=MODEL_SPARSE, openai_client=openai_client)

In [16]:
print("Results for OpenAI model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_openai_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_openai_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_openai_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_openai_sparse/n, 2)}")

Results for OpenAI model (Hybrid):
Hit Rate @ 5: 0.78
MRR @ 5: 0.58
Recall @ 5: 0.71
NDCG @ 5: 0.58


In [17]:
print("Results for OpenAI model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_sparse/n, 2)}")

Results for OpenAI model (Hybrid):
Hit Rate @ 5: 0.86
MRR @ 5: 0.78
Recall @ 5: 0.78
NDCG @ 5: 0.74


## Query Transform + Hybrid

In [12]:
total_hit_openai_transform_hybrid, total_mrr_openai_transform_hybrid, total_recall_openai_transform_hybrid, total_ndcg_openai_transform_hybrid, \
total_hit_reranked_openai_transform_hybrid, total_mrr_reranked_openai_transform_hybrid, total_recall_reranked_openai_transform_hybrid, total_ndcg_reranked_openai_transform_hybrid = functions_for_experiments.get_metrics_query_transform(None, COLLECTION_OPENAI, openai_client=openai_client, sparse_model=MODEL_SPARSE)

In [13]:
print("Results for OpenAI model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_openai_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_openai_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_openai_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_openai_transform_hybrid/n, 2)}")

Results for OpenAI model (Query Transform + Hybrid):
Hit Rate @ 5: 0.78
MRR @ 5: 0.6
Recall @ 5: 0.7
NDCG @ 5: 0.6


In [14]:
print("Results for OpenAI model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_openai_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_openai_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_openai_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_openai_transform_hybrid/n, 2)}")

Results for OpenAI model (Query Transform + Hybrid):
Hit Rate @ 5: 0.86
MRR @ 5: 0.78
Recall @ 5: 0.77
NDCG @ 5: 0.74
